In [41]:
from pyspark.ml.tuning import CrossValidatorModel
from pymongo import MongoClient
from pyspark.ml.feature import StringIndexer, VectorAssembler, MinMaxScaler, Imputer
from pyspark.sql import SparkSession
from pyspark.ml.classification import RandomForestClassificationModel


### MongoDB connection

In [42]:
mongo_uri = "mongodb://localhost:27017"
db_name = "heart"
collection_name = "heart-data"
output_collection_name = "prediction"

### MongoDB client for listening to changes

In [43]:
client = MongoClient(mongo_uri)
db = client[db_name]
collection = db[collection_name]

### Spark Session

In [44]:
spark = SparkSession.builder \
    .appName("RealTimePrediction") \
    .config("spark.mongodb.output.uri", f"{mongo_uri}/{db_name}.{output_collection_name}") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.1.1") \
    .getOrCreate()

### Loading the model

In [45]:
model = RandomForestClassificationModel.load("random_forest_model")

### Make prediction

In [46]:
def process_new_record(new_document):
    columns = ["age", "gender", "chestpaintype", "restingBP", "cholestrol", "fastingbloodsugar", "restingECG", "maxheartrate", "exerciseangia", "oldpeak", "slope", "target"]

    assembler = VectorAssembler(inputCols=columns[:-1], outputCol="features")
    new_data = [(new_document["age"], new_document["gender"], new_document["chestpaintype"], new_document["restingBP"], new_document["cholestrol"], new_document["fastingbloodsugar"], new_document["restingECG"], new_document["maxheartrate"], new_document["exerciseangia"], new_document["oldpeak"], new_document["slope"])]
    new_df = spark.createDataFrame(new_data, columns[:-1])
    new_df = assembler.transform(new_df)
    prediction = model.transform(new_df)
    prediction.show()

### Test 


In [47]:


# new_document = {
#   "_id": {
#     "$oid": "675215ce61f6e969b9c327c8"
#   },
#   "age": 37,
#   "gender": 1,
#   "chestpaintype": 0,
#   "restingBP": 198,
#   "cholestrol": 392,
#   "fastingbloodsugar": 1,
#   "restingECG": 2,
#   "maxheartrate": 219,
#   "exerciseangia": 0,
#   "oldpeak": 2.6,
#   "slope": 3
# }
# process_new_record(new_document)

### listening for new records

In [48]:
with collection.watch() as stream:
    print("Listening for new records...")
    for change in stream:
        if change["operationType"] == "insert":
            new_document = change["fullDocument"]
            process_new_record(new_document)

Listening for new records...
+---+------+-------------+---------+----------+-----------------+----------+------------+-------------+-------+-----+--------------------+-------------+-----------+----------+
|age|gender|chestpaintype|restingBP|cholestrol|fastingbloodsugar|restingECG|maxheartrate|exerciseangia|oldpeak|slope|            features|rawPrediction|probability|prediction|
+---+------+-------------+---------+----------+-----------------+----------+------------+-------------+-------+-----+--------------------+-------------+-----------+----------+
| 49|     0|            0|       94|         0|                1|         1|         144|            0|    2.0|    2|[49.0,0.0,0.0,94....|   [6.0,14.0]|  [0.3,0.7]|       1.0|
+---+------+-------------+---------+----------+-----------------+----------+------------+-------------+-------+-----+--------------------+-------------+-----------+----------+

+---+------+-------------+---------+----------+-----------------+----------+------------+-

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "C:\Users\ayman\AppData\Roaming\Python\Python39\site-packages\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "C:\Users\ayman\AppData\Roaming\Python\Python39\site-packages\py4j\clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "c:\python\lib\socket.py", line 704, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassificationModel

# Initialize Spark session
spark = SparkSession.builder \
    .appName("KafkaSparkStreamingPrediction") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.2.4") \
    .getOrCreate()

try:
    # Load the pre-trained Random Forest model
    model = RandomForestClassificationModel.load("zrandom_forest_model")
except Exception as e:
    print(f"Error loading model: {e}")
    spark.stop()
    exit(1)

# Define the schema for incoming data
schema = StructType([
    StructField("age", IntegerType(), True),
    StructField("gender", IntegerType(), True),
    StructField("chestpaintype", IntegerType(), True),
    StructField("restingBP", IntegerType(), True),
    StructField("cholestrol", IntegerType(), True),
    StructField("fastingbloodsugar", IntegerType(), True),
    StructField("restingECG", IntegerType(), True),
    StructField("maxheartrate", IntegerType(), True),
    StructField("exerciseangia", IntegerType(), True),
    StructField("oldpeak", DoubleType(), True),  # Changed to DoubleType
    StructField("slope", IntegerType(), True),
])

# Read data from Kafka
kafka_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "heart") \
    .option("startingOffsets", "latest") \
    .load()

# Parse the Kafka JSON messages
kafka_value_df = kafka_df.selectExpr("CAST(value AS STRING)") \
    .select(from_json("value", schema).alias("data")) \
    .select("data.*")

# Define features and assembler
columns = ["age", "gender", "chestpaintype", "restingBP", "cholestrol",
           "fastingbloodsugar", "restingECG", "maxheartrate", "exerciseangia",
           "oldpeak", "slope"]

assembler = VectorAssembler(inputCols=columns, outputCol="features")

# Assemble features
feature_df = assembler.transform(kafka_value_df)

# Make predictions
predictions = model.transform(feature_df)

# Write predictions to console for testing
query = predictions.select("age", "gender", "prediction").writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", "false") \
    .option("numRows", 10) \
    .start()

try:
    query.awaitTermination()
except KeyboardInterrupt:
    print("Stopping the streaming query...")
    query.stop()
    spark.stop()


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassificationModel

# Initialize Spark session
spark = SparkSession.builder \
    .appName("KafkaSparkStreamingPrediction") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.2.4") \
    .getOrCreate()

# Load the pre-trained Random Forest model
model = RandomForestClassificationModel.load("zrandom_forest_model")

# Define the schema for incoming data
schema = StructType([
    StructField("age", IntegerType(), True),
    StructField("gender", IntegerType(), True),
    StructField("chestpaintype", IntegerType(), True),
    StructField("restingBP", IntegerType(), True),
    StructField("cholestrol", IntegerType(), True),
    StructField("fastingbloodsugar", IntegerType(), True),
    StructField("restingECG", IntegerType(), True),
    StructField("maxheartrate", IntegerType(), True),
    StructField("exerciseangia", IntegerType(), True),
    StructField("oldpeak", DoubleType(), True),
    StructField("slope", IntegerType(), True),
])

# Read data from Kafka
kafka_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "heart") \
    .option("startingOffsets", "latest") \
    .load()

# Parse the Kafka JSON messages
parsed_stream = kafka_stream.selectExpr("CAST(value AS STRING)") \
    .select(from_json("value", schema).alias("data")) \
    .select("data.*")

# Define features and assembler
feature_columns = ["age", "gender", "chestpaintype", "restingBP", "cholestrol",
                   "fastingbloodsugar", "restingECG", "maxheartrate", "exerciseangia",
                   "oldpeak", "slope"]

assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")

# Function to process rows in each batch
def process_batch(batch_df, batch_id):
    # Assemble features
    batch_df = assembler.transform(batch_df)
    
    # Make predictions
    predictions = model.transform(batch_df)
    
    # Collect rows and print each prediction (row by row)
    rows = predictions.select(*feature_columns, "prediction").collect()
    for row in rows:
        print(f"Age: {row['age']}, Gender: {row['gender']}, Chest Pain Type: {row['chestpaintype']}, "
              f"Resting BP: {row['restingBP']}, Cholesterol: {row['cholestrol']}, Fasting Blood Sugar: {row['fastingbloodsugar']}, "
              f"Resting ECG: {row['restingECG']}, Max Heart Rate: {row['maxheartrate']}, Exercise Angina: {row['exerciseangia']}, "
              f"Old Peak: {row['oldpeak']}, Slope: {row['slope']}, Prediction: {row['prediction']}")

# Write the streaming query with foreachBatch
query = parsed_stream.writeStream \
    .outputMode("append") \
    .foreachBatch(process_batch) \
    .start()

# Wait for the streaming query to terminate
query.awaitTermination()


Age: 63, Gender: 1, Chest Pain Type: 0, Resting BP: 184, Cholesterol: 193, Fasting Blood Sugar: 0, Resting ECG: 1, Max Heart Rate: 192, Exercise Angina: 1, Old Peak: 3.7, Slope: 3, Prediction: 1.0
Age: 48, Gender: 0, Chest Pain Type: 0, Resting BP: 134, Cholesterol: 135, Fasting Blood Sugar: 0, Resting ECG: 0, Max Heart Rate: 165, Exercise Angina: 0, Old Peak: 3.3, Slope: 2, Prediction: 1.0
Age: 44, Gender: 1, Chest Pain Type: 2, Resting BP: 147, Cholesterol: 122, Fasting Blood Sugar: 0, Resting ECG: 2, Max Heart Rate: 101, Exercise Angina: 0, Old Peak: 5.0, Slope: 3, Prediction: 1.0
Age: 42, Gender: 0, Chest Pain Type: 3, Resting BP: 187, Cholesterol: 0, Fasting Blood Sugar: 1, Resting ECG: 1, Max Heart Rate: 135, Exercise Angina: 0, Old Peak: 3.3, Slope: 2, Prediction: 1.0
Age: 45, Gender: 0, Chest Pain Type: 2, Resting BP: 143, Cholesterol: 0, Fasting Blood Sugar: 0, Resting ECG: 2, Max Heart Rate: 102, Exercise Angina: 1, Old Peak: 1.0, Slope: 2, Prediction: 1.0
Age: 51, Gender: 0,

In [23]:
from pyspark.sql import SparkSession

# Initialize Spark session with Kafka dependencies

scala_version = '2.12'
spark_version = '3.2.4'
# TODO: Ensure match above values match the correct versions
packages = [
    f'org.apache.spark:spark-sql-kafka-0-10_{scala_version}:{spark_version}',
    'org.apache.kafka:kafka-clients:3.2.4'
]
spark = SparkSession.builder\
   .appName("kafka-example")\
   .config("spark.jars.packages", ",".join(packages))\
   .getOrCreate()

# Define Kafka parameters
kafka_bootstrap_servers = "localhost:9092"  # Replace with your Kafka broker(s)
kafka_topic = "your_topic_name"  # Replace with your topic name

# Read data from Kafka
df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers) \
    .option("subscribe", kafka_topic) \
    .load()

# Convert key and value columns from binary to string
df = df.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)")

# Define output sink (e.g., console)
query = df.writeStream \
    .outputMode("append") \
    .format("console") \
    .start()

# Start the stream and await termination
query.awaitTermination()


AnalysisException:  Failed to find data source: kafka. Please deploy the application as per the deployment section of "Structured Streaming + Kafka Integration Guide".        

In [26]:

from pyspark.sql.functions import expr
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, sum, avg
from pyspark.sql.types import StructType, StringType


scala_version = '2.12'
spark_version = '3.2.4'
# TODO: Ensure match above values match the correct versions
packages = [
    f'org.apache.spark:spark-sql-kafka-0-10_{scala_version}:{spark_version}',
    'org.apache.kafka:kafka-clients:3.2.4'
]
spark = SparkSession.builder\
   .appName("kafka-example")\
   .config("spark.jars.packages", ",".join(packages))\
   .getOrCreate()

topic = 'heart'

kafka_params = {"kafka.bootstrap.servers": "localhost:9092", "subscribe": "topic", "startingOffsets": "earliest"}

from pyspark.sql.types import StructType, StructField, IntegerType, StringType
schema = StructType([
    StructField("age", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("chestpaintype", StringType(), True),
    StructField("restingBP", IntegerType(), True),
    StructField("cholestrol", IntegerType(), True),
    StructField("fastingbloodsugar", IntegerType(), True),
    StructField("restingECG", StringType(), True),
    StructField("maxheartrate", IntegerType(), True),
    StructField("exerciseangia", StringType(), True),
    StructField("oldpeak", IntegerType(), True),
    StructField("slope", StringType(), True)
])

# Read data from Kafka stream
kafka_stream = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "heart") \
    .option("startingOffsets", "earliest") \
    .load()

# Extract data from Kafka message value
df = kafka_stream.selectExpr("CAST(value AS STRING) as json") \
    .select(from_json("json", json_schema).alias("data")) \
    .select("data.*")



# Write the results to the console
console_query1 = df \
    .writeStream \
    .outputMode("update") \
    .format("console") \
    .start()



console_query1.awaitTermination()





AnalysisException:  Failed to find data source: kafka. Please deploy the application as per the deployment section of "Structured Streaming + Kafka Integration Guide".        

In [ ]:
from pyspark.sql import SparkSession

# Create a SparkSession
spark = SparkSession.builder \
    .appName("KafkaStructuredStreaming") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.2.4") \
    .getOrCreate()

# Define Kafka parameters
kafka_bootstrap_servers = "localhost:9092"  # Replace with your Kafka broker's address
kafka_topic = "processed"  # Replace with your Kafka topic

# Create a DataFrame representing the stream of input lines from Kafka
kafka_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_bootstrap_servers) \
    .option("subscribe", kafka_topic) \
    .load()

# Convert Kafka's 'value' column to string
kafka_df = kafka_df.selectExpr("CAST(value AS STRING)")

# Function to process each micro-batch and display data in Jupyter Notebook
def process_batch(df, epoch_id):
    # This function will run on each micro-batch
    df.show(truncate=False)  # Display the DataFrame in Jupyter

# Start the streaming query using foreachBatch to show data in the notebook
query = kafka_df.writeStream \
    .foreachBatch(process_batch) \
    .outputMode("append") \
    .start()

# Await termination to keep the query running
query.awaitTermination()

# Stop the stream when done (you can stop manually or by calling query.stop())
query.stop()


+-----+
|value|
+-----+
+-----+

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                                                          |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{"age": 46, "gender": 0, "chestpaintype": 0, "restingBP": 137, "cholestrol": 192, "fastingbloodsugar": 1, "restingECG": 1, "maxheartrate": 191, "exerciseangia": 0, "oldpeak": 3.7, "slope": 3}|
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [1]:
from pyspark.sql import SparkSession

# Initialize Spark session with Kafka dependency
spark = SparkSession.builder \
    .appName("KafkaSparkStreaming") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.2.4") \
    .getOrCreate()




In [2]:

# Read from Kafka topic with earliest offset
kafka_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "heart") \
    .option("startingOffsets", "earliest") \
    .load()

In [3]:

# Convert Kafka binary key and value to strings
kafka_df = kafka_stream.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)")

In [4]:


# Write the stream to an in-memory table
query = kafka_df.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("kafka_table") \
    .start()

# Allow some time for the stream to process data (e.g., 10 seconds)
import time
time.sleep(10)

# Query and display the data in the notebook
if spark.sql("SELECT * FROM kafka_table").count() > 0:  # Check if data is available
    display_data = spark.sql("SELECT * FROM kafka_table")
    display_data.show(truncate=False)  # Print full data
else:
    print("No data available in Kafka topic yet.")

# Stop the stream after querying (optional)
query.stop()

+----+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|key |value                                                                                                                                                                                          |
+----+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|null|{"age": 75, "gender": 0, "chestpaintype": 1, "restingBP": 132, "cholestrol": 347, "fastingbloodsugar": 0, "restingECG": 1, "maxheartrate": 201, "exerciseangia": 1, "oldpeak": 0.6, "slope": 2}|
|null|{"age": 52, "gender": 0, "chestpaintype": 0, "restingBP": 129, "cholestrol": 367, "fastingbloodsugar": 1, "restingECG": 0, "maxheartrate": 108, "exerciseangia": 1, "oldpeak": 2.8, "slope": 2}|
|null